# Phase 5 · Data analysis — Legal Statute Identification (LSI, multi-label)

Computes the statistics needed to design the Phase 5 models and writes two small files to send back:
- `P5_analysis/p5_stats.json` — sizes, statute-label distribution (how many statutes, how rare), labels per case, co-occurrence, document lengths in characters / words / BPE tokens, overlaps between splits (leakage check), and whether statute references ("section 302", "article 21", …) are still in the text, where they occur and how strongly they go with each label.
- `P5_analysis/p5_samples.json` — a few example cases (beginning, end and text around statute references) with their labels.

**Kaggle setup:** CPU is enough (no GPU) · Add-ons → Secrets: **`HF`** · Internet on · Add Input → **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer). Expected time ≈ 15–25 minutes (tokenising every document dominates). Download both files from the Output panel (`P5_analysis/`) and send them.

In [ ]:
# Cell 1 — Imports and output folder.
import json, glob, random, hashlib, re
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import numpy as np
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

DATA_DIR = Path("/kaggle/working/data")
OUT_DIR = Path("/kaggle/working/P5_analysis")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(42)
print("Outputs:", OUT_DIR)

In [ ]:
# Cell 2 — Download Phase_5_LSI from Hugging Face using the Kaggle secret "HF".
snapshot_download(repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                  token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_5_LSI/**"])
P5_DIR = DATA_DIR / "Phase_5_LSI"
for f in sorted(P5_DIR.rglob("*")):
    if f.is_file():
        print(f"{f.relative_to(P5_DIR)}: {f.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Cell 3 — Load the three splits and show their record structure (field names and types).
def load(path):
    data = json.load(open(path))
    return data if isinstance(data, list) else [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v}
                                                 for k, v in data.items()]

splits = {"train": load(P5_DIR / "Train/train.json"), "dev": load(P5_DIR / "Train/dev.json"),
          "test": load(P5_DIR / "Test/test.json")}
stats = {"n_cases": {name: len(docs) for name, docs in splits.items()}, "fields": {}}
for name, docs in splits.items():
    stats["fields"][name] = {k: type(v).__name__ for k, v in docs[0].items()}
    print(f"{name}: {len(docs)} cases | fields: {stats['fields'][name]}")
    print("   example:", {k: (v[:80] + "…" if isinstance(v, str) and len(v) > 80 else v) for k, v in docs[0].items()})
LABEL_KEY, TEXT_KEY = "labels", "text"
assert all(LABEL_KEY in d for d in splits["train"]), "train records have no 'labels' field"

In [ ]:
# Cell 4 — Statute labels: how many distinct IDs, their range, labels per case, and IDs seen only in dev.
label_sets = {name: [sorted(set(int(x) for x in d[LABEL_KEY])) for d in splits[name]] for name in ("train", "dev")}
freq = {name: Counter(l for ls in label_sets[name] for l in ls) for name in ("train", "dev")}
all_ids = sorted(set(freq["train"]) | set(freq["dev"]))
per_case = {name: [len(ls) for ls in label_sets[name]] for name in ("train", "dev")}
dup_in_record = sum(len(d[LABEL_KEY]) != len(set(d[LABEL_KEY])) for d in splits["train"])
stats["labels"] = {
    "n_distinct_train": len(freq["train"]), "n_distinct_dev": len(freq["dev"]), "n_distinct_all": len(all_ids),
    "min_id": min(all_ids), "max_id": max(all_ids), "ids_contiguous": all_ids == list(range(min(all_ids), max(all_ids) + 1)),
    "ids_in_dev_not_train": sorted(set(freq["dev"]) - set(freq["train"])),
    "ids_in_train_not_dev": sorted(set(freq["train"]) - set(freq["dev"])),
    "records_with_duplicate_ids_train": dup_in_record,
    "cases_with_no_label": {name: sum(n == 0 for n in per_case[name]) for name in per_case},
    "labels_per_case": {name: {"mean": round(float(np.mean(v)), 3), "max": int(max(v)),
                               "distribution": dict(sorted(Counter(v).items()))} for name, v in per_case.items()},
}
print(json.dumps({k: v for k, v in stats["labels"].items() if k != "labels_per_case"}, indent=1))
for name, v in stats["labels"]["labels_per_case"].items():
    print(f"{name}: labels per case mean {v['mean']}, max {v['max']}, distribution {v['distribution']}")

In [ ]:
# Cell 5 — How rare are the statutes? Frequency of every ID (train and dev) and rare-label counts (Macro-F1 averages over IDs).
counts = np.array([freq["train"][i] for i in all_ids])
stats["label_frequency"] = {
    "train": {str(i): freq["train"][i] for i in all_ids}, "dev": {str(i): freq["dev"][i] for i in all_ids},
    "train_percentiles": {f"p{p}": int(np.percentile(counts, p)) for p in (0, 10, 25, 50, 75, 90, 100)},
    "ids_with_train_count_below": {str(k): int((counts < k).sum()) for k in (5, 10, 20, 50, 100)},
    "ids_with_dev_count_below": {str(k): int(sum(freq["dev"][i] < k for i in all_ids)) for k in (1, 3, 5, 10)},
    "share_of_label_mass_top10pct_ids": round(float(np.sort(counts)[::-1][:max(1, len(counts) // 10)].sum() / counts.sum()), 3),
}
order = sorted(all_ids, key=lambda i: -freq["train"][i])
print("Most frequent (id: train / dev):", [(i, freq["train"][i], freq["dev"][i]) for i in order[:15]])
print("Rarest (id: train / dev):", [(i, freq["train"][i], freq["dev"][i]) for i in order[-15:]])
print("Train-count percentiles:", stats["label_frequency"]["train_percentiles"])
print("IDs with train count below k:", stats["label_frequency"]["ids_with_train_count_below"])
print("IDs with dev count below k:", stats["label_frequency"]["ids_with_dev_count_below"])
print("Share of all label occurrences taken by the top 10% IDs:", stats["label_frequency"]["share_of_label_mass_top10pct_ids"])

In [ ]:
# Cell 6 — Label co-occurrence: most common statute pairs and the strongest dependencies P(B | A).
pairs = Counter(p for ls in label_sets["train"] for p in combinations(ls, 2))
dependent = []
for (a, b), n in pairs.items():
    for x, y in ((a, b), (b, a)):
        if freq["train"][x] >= 20:
            dependent.append((round(n / freq["train"][x], 3), x, y, n))
dependent.sort(reverse=True)
combos = Counter(tuple(ls) for ls in label_sets["train"])
stats["cooccurrence"] = {
    "top_pairs": [[a, b, n] for (a, b), n in pairs.most_common(25)],
    "strongest_P_B_given_A (A with >=20 cases)": [[x, y, p, n] for p, x, y, n in dependent[:25]],
    "distinct_label_sets_train": len(combos), "top_label_sets": [[list(k), v] for k, v in combos.most_common(15)],
    "dev_label_sets_unseen_in_train": sum(tuple(ls) not in combos for ls in label_sets["dev"]),
}
print("Top pairs (a, b, count):", stats["cooccurrence"]["top_pairs"][:12])
print("Strongest P(B|A) (A, B, P, count):", stats["cooccurrence"]["strongest_P_B_given_A (A with >=20 cases)"][:12])
print(f"Distinct label sets in train: {len(combos)} | dev cases whose exact set never occurs in train: "
      f"{stats['cooccurrence']['dev_label_sets_unseen_in_train']} of {len(label_sets['dev'])}")

In [ ]:
# Cell 7 — Document lengths: characters and words per split (percentiles), very short documents.
def pct(xs, ps=(0, 10, 25, 50, 75, 90, 95, 99, 100)):
    arr = np.array(xs)
    return {f"p{p}": int(np.percentile(arr, p)) for p in ps} | {"mean": int(arr.mean())}

stats["chars"], stats["words"], stats["short_docs"] = {}, {}, {}
for name, docs in splits.items():
    stats["chars"][name] = pct([len(d[TEXT_KEY]) for d in docs])
    words = [len(d[TEXT_KEY].split()) for d in docs]
    stats["words"][name] = pct(words)
    stats["short_docs"][name] = sum(w < 50 for w in words)
    print(f"{name}: words {stats['words'][name]} | docs under 50 words: {stats['short_docs'][name]}")
lower = {name: round(float(np.mean([d[TEXT_KEY] == d[TEXT_KEY].lower() for d in docs[:2000]])), 3) for name, docs in splits.items()}
stats["share_all_lowercase"] = lower
print("Share of documents that are entirely lower-case:", lower)

In [ ]:
# Cell 8 — BPE-token lengths with the Phase-1 tokenizer and the number of 512-token chunks per case.
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
stats["tokens"], stats["chunks_512"], stats["tokens_per_word"] = {}, {}, {}
for name, docs in splits.items():
    lengths = []
    for start in range(0, len(docs), 256):
        lengths += [len(e.ids) for e in tokenizer.encode_batch([d[TEXT_KEY] for d in docs[start:start + 256]])]
    stats["tokens"][name] = pct(lengths)
    stats["chunks_512"][name] = pct([max(1, -(-n // 510)) for n in lengths])
    stats["tokens_per_word"][name] = round(sum(lengths) / max(1, sum(len(d[TEXT_KEY].split()) for d in docs)), 3)
    print(f"{name}: tokens {stats['tokens'][name]}")
    print(f"      512-token chunks per case {stats['chunks_512'][name]} | total tokens {sum(lengths):,}")

In [ ]:
# Cell 9 — Overlaps between splits (leakage check): shared ids, identical texts, and whether test ⊂ dev as in Phase 4.
def text_hash(t):
    return hashlib.md5(" ".join(t.lower().split()).encode()).hexdigest()

ids = {name: {str(d["id"]) for d in docs} for name, docs in splits.items()}
hashes = {name: Counter(text_hash(d[TEXT_KEY]) for d in docs) for name, docs in splits.items()}
stats["overlap"] = {
    "ids_train_dev": len(ids["train"] & ids["dev"]), "ids_train_test": len(ids["train"] & ids["test"]),
    "ids_dev_test": len(ids["dev"] & ids["test"]),
    "text_train_dev": len(set(hashes["train"]) & set(hashes["dev"])),
    "text_train_test": len(set(hashes["train"]) & set(hashes["test"])),
    "text_dev_test": len(set(hashes["dev"]) & set(hashes["test"])),
    "duplicate_texts_within_train": sum(c - 1 for c in hashes["train"].values() if c > 1),
}
by_hash = defaultdict(set)
for d, ls in zip(splits["train"], label_sets["train"]):
    by_hash[text_hash(d[TEXT_KEY])].add(tuple(ls))
stats["overlap"]["train_duplicates_with_different_labels"] = sum(len(v) > 1 for v in by_hash.values())
test_in_dev = {text_hash(d[TEXT_KEY]) for d in splits["test"]} & set(hashes["dev"])
stats["overlap"]["dev_clean_cases_if_test_texts_removed"] = sum(text_hash(d[TEXT_KEY]) not in test_in_dev for d in splits["dev"])
print(json.dumps(stats["overlap"], indent=1))

In [ ]:
# Cell 10 — Are statute references still in the text? Counts of "section N", "article N", "s. N", act names and
# placeholder/mask tokens, and where in the document (relative position) they occur.
PATTERNS = {"section_N": r"\bsec(?:tion|s?\.)?\s*\d+[a-z]?\b", "article_N": r"\bart(?:icle|\.)\s*\d+[a-z]?\b",
            "s_dot_N": r"\bs\.\s*\d+[a-z]?\b", "u_s_N": r"\bu/s\.?\s*\d+", "ipc": r"\bi\.?\s?p\.?\s?c\b|indian penal code",
            "crpc": r"\bcr\.?\s?p\.?\s?c\b|code of criminal procedure", "constitution": r"\bconstitution\b",
            "act_word": r"\bact\b"}
MASK_CANDIDATES = r"<[a-z_]+>|\[[a-z_]+\]|\b(?:xxx+|_{3,})\b"
sample = random.sample(splits["train"], min(3000, len(splits["train"])))
rates, positions, masks = {}, defaultdict(list), Counter()
for key, pat in PATTERNS.items():
    rx = re.compile(pat, re.I)
    hits = 0
    for d in sample:
        t = d[TEXT_KEY]
        found = [m.start() / max(len(t), 1) for m in rx.finditer(t)]
        hits += bool(found)
        positions[key] += found
    rates[key] = round(hits / len(sample), 3)
for d in sample:
    masks.update(m.lower() for m in re.findall(MASK_CANDIDATES, d[TEXT_KEY], re.I))
stats["statute_mentions_train_sample"] = {
    "share_of_docs_with_pattern": rates,
    "relative_position_deciles": {k: [int(x) for x in np.histogram(v, bins=10, range=(0, 1))[0]] for k, v in positions.items()},
    "mask_like_tokens_top": masks.most_common(20),
}
print("Share of documents containing each pattern:", rates)
print("Where 'section N' occurs (counts per tenth of the document):",
      stats["statute_mentions_train_sample"]["relative_position_deciles"]["section_N"])
print("Mask-like tokens:", masks.most_common(20))

In [ ]:
# Cell 11 — Does a mentioned "section N" point to a statute ID? For the 20 most frequent IDs, the section numbers whose
# presence most raises P(label) (lift), on all training cases. Strong lifts mean the text still names the statute.
SEC_RX = re.compile(r"\b(?:section|sec\.?|s\.|u/s\.?)\s*(\d+[a-z]?)\b", re.I)
doc_secs = [set(m.lower() for m in SEC_RX.findall(d[TEXT_KEY])) for d in splits["train"]]
sec_df = Counter(s for secs in doc_secs for s in secs)
n_docs = len(doc_secs)
association = {}
for lab in order[:20]:
    has = [lab in ls for ls in label_sets["train"]]
    base = sum(has) / n_docs
    together = Counter(s for secs, h in zip(doc_secs, has) if h for s in secs)
    scored = [(round((together[s] / sec_df[s]) / base, 2), s, together[s], sec_df[s]) for s in together if sec_df[s] >= 30]
    scored.sort(reverse=True)
    association[str(lab)] = {"base_rate": round(base, 4), "top_sections (lift, section, docs_with_both, docs_with_section)": scored[:5],
                             "share_of_label_docs_mentioning_best_section":
                             round(together[scored[0][1]] / sum(has), 3) if scored else None}
    print(f"ID {lab:>4} (base {base:.3f}): {scored[:3]}")
stats["section_label_association"] = association

In [ ]:
# Cell 12 — Save the statistics and example cases (first/last 1,000 characters and text around statute references).
samples = {}
def around_sections(t, k=3, width=150):
    return [t[max(0, m.start() - width):m.end() + width] for m in list(SEC_RX.finditer(t))[:k]]
for name in ("train", "dev"):
    chosen = random.sample(range(len(splits[name])), 4)
    samples[name] = [{"id": str(splits[name][i]["id"]), "labels": label_sets[name][i], "n_chars": len(splits[name][i][TEXT_KEY]),
                      "start": splits[name][i][TEXT_KEY][:1000], "end": splits[name][i][TEXT_KEY][-1000:],
                      "around_section_mentions": around_sections(splits[name][i][TEXT_KEY])} for i in chosen]
rare_ids = [i for i in order if freq["train"][i] >= 1][-3:]
samples["rare_label_cases"] = [{"id": str(d["id"]), "labels": ls, "start": d[TEXT_KEY][:600],
                                "around_section_mentions": around_sections(d[TEXT_KEY])}
                               for d, ls in zip(splits["train"], label_sets["train"]) if set(ls) & set(rare_ids)][:3]
samples["test"] = [{"id": str(d["id"]), "n_chars": len(d[TEXT_KEY]), "start": d[TEXT_KEY][:600]}
                   for d in random.sample(splits["test"], 2)]
json.dump(stats, open(OUT_DIR / "p5_stats.json", "w"), indent=2)
json.dump(samples, open(OUT_DIR / "p5_samples.json", "w"), indent=2)
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e3:.1f} KB")
print("Download both files from the Output panel (P5_analysis/) and send them.")